In [1]:
import numpy as np
import torch
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.parallel import DistributedDataParallel, DataParallel
from utils import get_sigma_time, get_sample_time, VESDE, get_config
from model import UNet3DModel
import matplotlib.pyplot as plt
from torch_ema import ExponentialMovingAverage
import logging
import os
import sys
from os.path import join
import argparse

In [70]:
from dataclasses import dataclass

@dataclass
class args:
    config = 'configs/config_dm_1900_2.json'
    disable_tqdm = False

In [71]:
config_filename = args.config
enable_tqdm = not args.disable_tqdm
config = get_config(config_filename)

In [72]:
Nside = config.data.image_size
#DEVICE = config.device
DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')


sigma_time = get_sigma_time(config.model.sigma_min, config.model.sigma_max)
sample_time = get_sample_time(config.model.sampling_eps, config.model.T)

cosmo_dir = config.model.cosmo_dir
data_path = join(config.model.workdir, cosmo_dir)
checkpoint_dir = join(data_path, config.model.checkpoint_dir)

In [73]:
# Build pytorch dataloaders
input_data = np.float32(np.load(join(data_path, 'observation.npy')))
print("Loaded shape:", input_data.shape)
label_data = np.float32(np.load(join(data_path, 'truth.npy')))
input_data = torch.from_numpy(input_data).to(DEVICE)
label_data = torch.from_numpy(label_data).to(DEVICE)
input_data = torch.unsqueeze(input_data, dim=1)
label_data = torch.unsqueeze(label_data, dim=1)

Loaded shape: (1, 128, 128, 128)


In [74]:
# Initialize score model
model = UNet3DModel(config)
#model = DataParallel(model)
model = model.to(DEVICE)

ema = ExponentialMovingAverage(model.parameters(), decay=config.model.ema_rate)

sde = VESDE(config.model.sigma_min, config.model.sigma_max, config.model.num_scales, config.model.T, config.model.sampling_eps)

In [75]:
# Check for existing checkpoint
checkpoint_path = join(checkpoint_dir, 'checkpoint.pth')
if os.path.isfile(checkpoint_path):
    loaded_state = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(loaded_state['model'], strict=False)
    ema.load_state_dict(loaded_state['ema'])
    logging.info(f"Loaded checkpoint from {checkpoint_path}.")
    print(f"Loaded checkpoint from {checkpoint_path}.")
else:
    logging.warning(f"No checkpoint found at {checkpoint_path}. Starting from scratch.")
    print(f"No checkpoint found at {checkpoint_path}. Starting from scratch.")

_ = model.eval()

Loaded checkpoint from run/cosmos_dm_1900_2/checkpoints/checkpoint.pth.


In [76]:
def one_step(x, t):
    t_vec = torch.ones(shape[0], device=DEVICE) * t
    model_output = model(torch.cat([x, input_data], dim=1), t_vec)
    x, x_mean = sde.update_fn(x, t_vec, model_output=model_output)
    return x, x_mean

print("input_data shape before tiling:", input_data.shape)


input_data = torch.tile(input_data, dims=(config.sampling.batch_size, 1, 1, 1, 1))
shape = (config.sampling.batch_size, 1, Nside, Nside, Nside)

input_data shape before tiling: torch.Size([1, 1, 128, 128, 128])


In [77]:
import torch
import numpy as np
import time
from tqdm import tqdm
# You may need to install fvcore: pip install fvcore
from fvcore.nn import FlopCountAnalysis

In [78]:
# Dummy inputs for FLOP calculation (do this once)
dummy_x0 = torch.randn((1, 1, Nside, Nside, Nside), device=DEVICE)
dummy_h = torch.randn((1, 1, Nside, Nside, Nside), device=DEVICE) # matching x shape
# Note: FlopCountAnalysis works best on standard modules; 
# if .predict is a complex custom method, this provides a close estimate.
t_vec = torch.ones(1, device=DEVICE)

flops_per_sample = FlopCountAnalysis(model, (torch.cat([dummy_x0, dummy_h], dim=1), t_vec.to(DEVICE))).total()

In [79]:
samples = []
print('Sampling begins.')

n_samples = 1

total_time = 0
peak_memory = 0
total_flops = 0

starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
starter.record()

for j in tqdm(
    # range(config.sampling.num_samples//config.sampling.batch_size),
    range(n_samples),
    disable=args.disable_tqdm
):
    with torch.no_grad(), ema.average_parameters():
        x = sde.prior_sampling(shape).to(DEVICE)
        timesteps = sde.timesteps.to(DEVICE)
        for i in tqdm(range(sde.N), disable=args.disable_tqdm):
            t = timesteps[i]
            x, x_mean = one_step(x, t)
        samples.append(x_mean.detach().cpu().numpy())
#     np.save(data_path + 'sample.npy', np.array(samples))
#     print(f'Finished {j+1}th round')

# print('Done sampling')
# np.save(data_path + 'sample.npy', np.array(samples))

ender.record()
torch.cuda.synchronize()

# Metrics calculation
total_time += starter.elapsed_time(ender) / 1000 # convert ms to seconds
peak_memory = max(peak_memory, torch.cuda.max_memory_allocated(DEVICE) / (1024**2)) # MB
total_flops += (flops_per_sample * n_samples)

Sampling begins.


100%|██████████| 1/1 [04:00<00:00, 240.87s/it]


In [80]:
count = 1

avg_time = total_time / (n_samples*count)
avg_flops = total_flops / (n_samples*count)

print(f"\n{'='*30}")
print(f"Benckmark Results (Avg per sample):")
print(f"Time: {avg_time:.4f} seconds")
print(f"FLOPs: {avg_flops / 1e9:.2f} GFLOPs")
print(f"Peak Memory: {peak_memory:.2f} MB")


Benckmark Results (Avg per sample):
Time: 240.8759 seconds
FLOPs: 1798.74 GFLOPs
Peak Memory: 19953.09 MB


# Diffusion

## 32 

Time: 27.8612 seconds
FLOPs: 47.67 GFLOPs
Peak Memory: 19818.03 MB

## 64

Time: 219.9999 seconds
FLOPs: 474.14 GFLOPs
Peak Memory: 19953.09 MB

## 128 

Time: 243.2822 seconds
FLOPs: 1798.74 GFLOPs
Peak Memory: 19953.09 MB